# One-Step Virtual Update Interference Probe (HiP-AD)

**Goal:** Measure task-to-task interference via a *one-step virtual update probe* at the **mini-batch** level (not single-sample).

For a fixed mini-batch $B$ and parameters $\theta$:

1. Compute baseline $L_{\text{plan}}(B, \theta)$.
2. For each probe task $i \in \{\text{det}, \text{map}, \text{motion}, \text{ego}, \text{plan}\}$:
   - Compute $g_i = \partial L_i(B, \theta) / \partial W_{\text{shared}}$
   - Virtual step: $\theta_i' = \theta - \alpha \cdot g_i$ (or normalized: $\theta - \alpha \cdot g_i / \|g_i\|$)
   - Recompute **on the same batch**: $L_{\text{plan}}(B, \theta_i')$
   - $\Delta L_{\text{plan} \leftarrow i} = L_{\text{plan}}(B, \theta_i') - L_{\text{plan}}(B, \theta)$
   - Immediately restore $\theta$.
3. Aggregate over N mini-batches: mean / std / harmful ratio.

**Interpretation:**

| $\Delta L_{\text{plan} \leftarrow i}$ | Meaning |
|---|---|
| $> 0$ | Task $i$'s local step is **harmful** to planning |
| $< 0$ | Task $i$'s local step is **helpful** to planning |
| $\approx 0$ | Weak local influence |

**Design principles:**
- Mini-batch gradients (matches real optimizer step).
- Plain SGD-style virtual step (no AdamW state mixing).
- Parameter restore after every probe — optimizer.step() never called.
- Same batch reused for before/after forward.
- Shared params match `analyze_gradient_conflict.py` (backbone / neck / norm / ffn / fc_before / fc_after). Note: backbone/neck default extraction is limited to decoder-side shared layers; override `SHARED_LAYERS` below if you want to include backbone/neck explicitly.


## 1. Configuration

Edit these values to match your run. All knobs are centralized here so the rest of the notebook is reusable.

In [1]:
# ---------------------------------------------------------------------------
# User-tunable configuration (equivalent to CLI args in a standalone script)
# ---------------------------------------------------------------------------
CONFIG_PATH     = '/home/yongjae/e2e/HiP-AD/projects/configs/experiments/E2_E1_stage2_18ep.py'
CHECKPOINT_PATH = '/home/yongjae/e2e/HiP-AD/ckpts/exp/E2_E1_stage2_18ep/iter_2344.pth'

# --- Match training-iteration data volume -----------------------------------
# E2_E1_stage2_18ep.py: num_gpus=2, batch_size=6 per GPU.
# Effective mini-batch per optimizer step = num_gpus * batch_size = 12.
# Probing runs on a single GPU, so BATCH_SIZE = 12 makes every probed
# mini-batch process the same data volume as one real training iteration.
TRAIN_NUM_GPUS    = 2
TRAIN_PER_GPU_BS  = 6
EFFECTIVE_BATCH   = TRAIN_NUM_GPUS * TRAIN_PER_GPU_BS  # = 12

NUM_BATCHES     = 20                  # How many mini-batches to probe
BATCH_SIZE      = EFFECTIVE_BATCH     # single-GPU batch == 1 training iteration worth of data
DEVICE          = 'cuda:0'
FP16            = True
SEED            = 42

SHARED_LAYERS   = ['backbone', 'neck', 'norm', 'ffn', 'fc_before', 'fc_after']
LAST_N_LAYERS   = None        # None -> all decoder layers
PARAM_SCOPE     = 'shared_only'   # 'shared_only' | 'all'
NO_SELECTIVE_EVAL = False     # True -> skip _selective_eval (use raw train mode)

FOCUS_TASK      = 'plan'
PROBE_TASKS     = ['det', 'map', 'motion', 'ego', 'plan']
VIRTUAL_LR      = 1e-3        # alpha for one-step probe
NORMALIZE_STEP  = False       # True -> normalize g / ||g|| before step
NORMALIZE_SCOPE = 'all'       # 'all' -> full shared-grad vector norm; 'group' -> per group norm

OUTPUT_DIR      = 'gradient_analysis_results/one_step_probe'
SAVE_PER_BATCH  = True
SAVE_JSON       = True
SAVE_CSV        = True

# ---------------------------------------------------------------------------
print(f"Config: {CONFIG_PATH}")
print(f"Checkpoint: {CHECKPOINT_PATH}")
print(f"Training setup: num_gpus={TRAIN_NUM_GPUS}, per-GPU batch={TRAIN_PER_GPU_BS} "
      f"-> effective batch = {EFFECTIVE_BATCH}")
print(f"Probe BATCH_SIZE (single-GPU) = {BATCH_SIZE} (matches training effective batch)")
print(f"Probe tasks: {PROBE_TASKS} | focus: {FOCUS_TASK}")
print(f"Virtual LR: {VIRTUAL_LR} | normalize: {NORMALIZE_STEP} ({NORMALIZE_SCOPE})")
print(f"Num batches: {NUM_BATCHES} | param_scope: {PARAM_SCOPE}")


Config: /home/yongjae/e2e/HiP-AD/projects/configs/experiments/E2_E1_stage2_18ep.py
Checkpoint: /home/yongjae/e2e/HiP-AD/ckpts/exp/E2_E1_stage2_18ep/iter_2344.pth
Training setup: num_gpus=2, per-GPU batch=6 -> effective batch = 12
Probe BATCH_SIZE (single-GPU) = 12 (matches training effective batch)
Probe tasks: ['det', 'map', 'motion', 'ego', 'plan'] | focus: plan
Virtual LR: 0.001 | normalize: False (all)
Num batches: 20 | param_scope: shared_only


## 2. Imports & helper reuse

We re-use the core helpers from `analyze_gradient_conflict.py` directly rather than duplicating them.

In [2]:
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "2,3"

import sys
import json
import math
import csv
import contextlib
import traceback
from functools import partial
from collections import OrderedDict, defaultdict
from typing import Dict, List, Tuple, Optional

import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import DataLoader

# Resolve project root (HiP-AD repo root = parent of tools/)
_NB_DIR = os.path.abspath(os.path.dirname('.'))
# When run from repo root: '.'; when run from tools/: parent
_CANDIDATES = [os.path.abspath('.'), os.path.abspath('..')]
PROJECT_ROOT = None
for c in _CANDIDATES:
    if os.path.isdir(os.path.join(c, 'projects')) and os.path.isdir(os.path.join(c, 'tools')):
        PROJECT_ROOT = c
        break
assert PROJECT_ROOT is not None, \
    "Could not locate HiP-AD project root. Run notebook from repo root or tools/."
os.chdir(PROJECT_ROOT)
if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)
TOOLS_DIR = os.path.join(PROJECT_ROOT, 'tools')
if TOOLS_DIR not in sys.path:
    sys.path.insert(0, TOOLS_DIR)

print(f"PROJECT_ROOT = {PROJECT_ROOT}")

# MMCV / MMDet
from mmcv import Config
from mmcv.runner import load_checkpoint, wrap_fp16_model
from mmcv.parallel import collate, MMDataParallel
from mmdet.models import build_detector

# HiP-AD plugin imports (registers custom modules)
import projects.mmdet3d_plugin  # noqa: F401
from projects.mmdet3d_plugin.datasets.builder import custom_build_dataset

# Reuse helpers from analyze_gradient_conflict.py (no duplication)
from analyze_gradient_conflict import (
    TASK_GROUPS,
    _sum_task_loss,
    _selective_eval,
    move_to_device,
    get_shared_parameters_grouped,
    compute_task_gradient_grouped,
    compute_cosine_similarity,
)

print("Reused helpers:",
      "TASK_GROUPS, _sum_task_loss, _selective_eval, move_to_device, "
      "get_shared_parameters_grouped, compute_task_gradient_grouped, compute_cosine_similarity")

PROJECT_ROOT = /home/yongjae/e2e/HiP-AD


/home/yongjae/miniconda3/envs/hipad/lib/python3.8/site-packages/mmcv/__init__.py:20: UserWarning: On January 1, 2023, MMCV will release v2.0.0, in which it will remove components related to the training process and add a data transformation module. In addition, it will rename the package names mmcv to mmcv-lite and mmcv-full to mmcv. See https://github.com/open-mmlab/mmcv/blob/master/docs/en/compatibility.md for more details.
  warnings.warn(


Use flash_attn_varlen_kvpacked_func
Reused helpers: TASK_GROUPS, _sum_task_loss, _selective_eval, move_to_device, get_shared_parameters_grouped, compute_task_gradient_grouped, compute_cosine_similarity


## 3. Core probe functions

New, probe-specific logic. These are the functions *added* on top of what `analyze_gradient_conflict.py` already provides.

- `extract_probe_params`: picks the parameter set to virtually update (shared_only or all).
- `backup_params` / `restore_params`: checkpoint & rollback of parameter tensors (no optimizer state).
- `apply_virtual_step`: applies `p ← p − α · g` in-place (raw or normalized).
- `forward_and_get_losses`: single forward pass on a cached batch; returns loss dict and planning-loss tensor **with grad graph retained** when needed for gradient computation.
- `compute_planning_loss_value`: pure `no_grad` forward for the *recompute* pass after a virtual step.
- `run_probe_on_batch`: full before / per-task-probe / after pipeline for one mini-batch.
- `run_full_probe`: outer loop over `NUM_BATCHES` mini-batches.

In [3]:
EPS = 1e-12


def extract_probe_params(
    model: nn.Module,
    shared_layers: List[str],
    last_n_layers: Optional[int],
    param_scope: str,
) -> Tuple[OrderedDict, OrderedDict]:
    """Return (param_groups, group_meta). Matches the shape used by
    compute_task_gradient_grouped so we can reuse it directly.

    param_scope:
      - 'shared_only' : delegate to get_shared_parameters_grouped
      - 'all'         : every requires_grad param, in a single '_all' group
    """
    if param_scope == 'shared_only':
        param_groups, group_meta = get_shared_parameters_grouped(
            model, shared_layers, last_n_layers
        )
        return param_groups, group_meta

    if param_scope == 'all':
        params = OrderedDict()
        for name, p in model.named_parameters():
            if p.requires_grad:
                params[name] = p
        param_groups = OrderedDict([('_all_params', params)])
        group_meta = OrderedDict([('_all_params', {
            'decoder_idx': -1, 'op_type': 'all', 'global_layer_idx': -1,
        })])
        return param_groups, group_meta

    raise ValueError(f"Unknown param_scope: {param_scope}")


def flatten_param_group(param_groups: OrderedDict) -> List[nn.Parameter]:
    """Ordered flat list of parameters across all groups (matches
    compute_task_gradient_grouped's internal ordering)."""
    out: List[nn.Parameter] = []
    for _, params in param_groups.items():
        out.extend(params.values())
    return out


def backup_params(params: List[nn.Parameter]) -> List[torch.Tensor]:
    """CPU-side clone of each parameter tensor. CPU to avoid doubling GPU memory."""
    return [p.detach().clone() for p in params]


def restore_params(params: List[nn.Parameter], backup: List[torch.Tensor]) -> None:
    """In-place restore — no optimizer state involved."""
    with torch.no_grad():
        for p, b in zip(params, backup):
            p.data.copy_(b.data)


def apply_virtual_step(
    params: List[nn.Parameter],
    grad_flat: torch.Tensor,
    lr: float,
    normalize: bool,
    normalize_scope: str,
    group_boundaries: Optional[List[Tuple[int, int]]] = None,
) -> float:
    """In-place: p <- p - lr * g (or normalized variant). Returns the gradient
    norm actually used (for logging).

    normalize_scope:
      'all'   -> single shared-grad vector norm
      'group' -> per-group norm (needs group_boundaries: list of (start, end))
    """
    used_norm = float(torch.norm(grad_flat).item())

    if normalize and normalize_scope == 'all':
        g_norm = torch.norm(grad_flat) + EPS
        step_vec = grad_flat / g_norm
    elif normalize and normalize_scope == 'group':
        assert group_boundaries is not None, "group normalize requires boundaries"
        step_vec = grad_flat.clone()
        for (s, e) in group_boundaries:
            sub_norm = torch.norm(step_vec[s:e]) + EPS
            step_vec[s:e] = step_vec[s:e] / sub_norm
    else:
        step_vec = grad_flat

    with torch.no_grad():
        cursor = 0
        for p in params:
            n = p.numel()
            g_slice = step_vec[cursor:cursor + n].view_as(p)
            p.data.add_(g_slice.to(p.device, dtype=p.dtype), alpha=-lr)
            cursor += n
    return used_norm


def group_boundaries_from(param_groups: OrderedDict) -> List[Tuple[int, int]]:
    bounds = []
    cursor = 0
    for _, params in param_groups.items():
        n = sum(p.numel() for p in params.values())
        bounds.append((cursor, cursor + n))
        cursor += n
    return bounds


In [4]:
def _sum_task_loss_nograd(loss_dict: Dict[str, torch.Tensor], task_name: str) -> Optional[torch.Tensor]:
    """Variant of _sum_task_loss that does NOT require requires_grad.

    Used for the 'after virtual step' recompute, which we run under torch.no_grad()
    so the task losses will not have grad. The original _sum_task_loss filters
    those out.
    """
    from analyze_gradient_conflict import match_loss_key
    prefixes = TASK_GROUPS.get(task_name, [])
    task_loss = None
    for key, val in loss_dict.items():
        if match_loss_key(key, prefixes) and isinstance(val, torch.Tensor):
            task_loss = val if task_loss is None else task_loss + val
    return task_loss


def clone_batch(data: Dict) -> Dict:
    """Deep-ish copy of a batch dict. mmcv DataContainer objects are kept as-is
    (they are already immutable for our purposes); tensors are cloned."""
    out = {}
    for k, v in data.items():
        if isinstance(v, torch.Tensor):
            out[k] = v.clone()
        elif isinstance(v, list):
            out[k] = [x.clone() if isinstance(x, torch.Tensor) else x for x in v]
        elif isinstance(v, dict):
            out[k] = clone_batch(v)
        else:
            out[k] = v
    return out


def forward_losses(model: nn.Module, data: Dict, device: str, fp16: bool,
                   selective_eval: bool) -> Dict[str, torch.Tensor]:
    """Single forward pass with the training loss heads. Keeps the grad graph.

    Caller owns the returned dict; must zero_grad afterwards.
    """
    data_on_dev = move_to_device(data, device)
    model.train()
    ctx = _selective_eval(model) if selective_eval else contextlib.nullcontext()
    with ctx:
        with torch.cuda.amp.autocast(enabled=fp16):
            img = data_on_dev.pop('img')
            outputs = model(img=img, **data_on_dev)
    if not isinstance(outputs, dict):
        raise RuntimeError("Model output is not a loss dict — check training mode.")
    return outputs


@torch.no_grad()
def forward_losses_nograd(model: nn.Module, data: Dict, device: str, fp16: bool,
                          selective_eval: bool) -> Dict[str, torch.Tensor]:
    """No-grad forward used for the recompute pass. Cheaper and leak-free."""
    data_on_dev = move_to_device(data, device)
    model.train()
    ctx = _selective_eval(model) if selective_eval else contextlib.nullcontext()
    with ctx:
        with torch.cuda.amp.autocast(enabled=fp16):
            img = data_on_dev.pop('img')
            outputs = model(img=img, **data_on_dev)
    return outputs


def extract_task_loss_values(loss_dict: Dict[str, torch.Tensor],
                             tasks: List[str]) -> Dict[str, float]:
    """Scalarize each task's summed loss. Works with or without grad."""
    out = {}
    for t in tasks:
        lt = _sum_task_loss_nograd(loss_dict, t)
        out[t] = float(lt.item()) if lt is not None else float('nan')
    return out


In [5]:
def run_probe_on_batch(
    model: nn.Module,
    data: Dict,
    param_groups: OrderedDict,
    probe_tasks: List[str],
    focus_task: str,
    device: str,
    fp16: bool,
    virtual_lr: float,
    normalize_step: bool,
    normalize_scope: str,
    selective_eval: bool,
    batch_idx: int,
) -> Dict:
    """One mini-batch probe:
      A) baseline forward with grad graph
      B) compute gradient for each probe task on shared params
      C) for each probe task: virtual step -> no_grad forward -> delta -> restore
    Returns a dict with baseline + per-probe-task deltas.
    """
    params_flat = flatten_param_group(param_groups)
    g_bounds = group_boundaries_from(param_groups)

    # --- Baseline: need grad graph to compute per-task gradients --------------
    data_cache = clone_batch(data)  # one cached copy used for ALL forward passes
    loss_dict = forward_losses(model, clone_batch(data_cache), device, fp16, selective_eval)
    baseline_task_losses = extract_task_loss_values(loss_dict, list(TASK_GROUPS.keys()))
    baseline_focus_loss = baseline_task_losses.get(focus_task, float('nan'))

    # --- Compute gradients for each probe task (reuse retain_graph trick) -----
    task_grads: Dict[str, Optional[torch.Tensor]] = {}
    tasks_to_compute = list(probe_tasks)
    if focus_task not in tasks_to_compute:
        tasks_to_compute.append(focus_task)  # need focus grad for cosine

    for i, task in enumerate(tasks_to_compute):
        retain = (i < len(tasks_to_compute) - 1)
        grad_dict = compute_task_gradient_grouped(
            model, loss_dict, task, param_groups, retain_graph=retain,
        )
        if grad_dict is None:
            task_grads[task] = None
        else:
            # Keep '_all' on CPU to free GPU mem between probes
            task_grads[task] = grad_dict['_all'].detach().cpu()

    # free baseline graph
    del loss_dict
    model.zero_grad(set_to_none=True)
    torch.cuda.empty_cache()

    focus_grad = task_grads.get(focus_task)

    # --- Per-probe virtual step --------------------------------------------------
    backup = backup_params(params_flat)
    per_probe: Dict[str, Dict] = {}

    for task in probe_tasks:
        g = task_grads.get(task)
        probe_record = {
            'grad_norm': float('nan'),
            'cos_vs_focus': float('nan'),
            'delta_focus': float('nan'),
            'rel_delta_focus': float('nan'),
            'focus_loss_after': float('nan'),
            'failed': False,
            'error': None,
        }

        if g is None or torch.norm(g).item() < EPS:
            probe_record['failed'] = True
            probe_record['error'] = 'zero_or_missing_gradient'
            per_probe[task] = probe_record
            continue

        # cosine with focus gradient (pre-step, on same cached grads)
        if focus_grad is not None:
            probe_record['cos_vs_focus'] = compute_cosine_similarity(g, focus_grad)

        try:
            # Apply virtual step
            g_device = g.to(params_flat[0].device)
            used_norm = apply_virtual_step(
                params_flat, g_device, virtual_lr,
                normalize=normalize_step, normalize_scope=normalize_scope,
                group_boundaries=g_bounds,
            )
            probe_record['grad_norm'] = used_norm

            # Recompute focus loss on SAME batch
            loss_after = forward_losses_nograd(
                model, clone_batch(data_cache), device, fp16, selective_eval,
            )
            focus_after = _sum_task_loss_nograd(loss_after, focus_task)
            focus_after_val = float(focus_after.item()) if focus_after is not None else float('nan')

            delta = focus_after_val - baseline_focus_loss
            probe_record['focus_loss_after'] = focus_after_val
            probe_record['delta_focus'] = delta
            probe_record['rel_delta_focus'] = delta / (abs(baseline_focus_loss) + EPS)

            del loss_after

        except Exception as e:
            probe_record['failed'] = True
            probe_record['error'] = f"{type(e).__name__}: {e}"
            traceback.print_exc()
        finally:
            # ALWAYS restore before the next probe — keep same θ₀
            restore_params(params_flat, backup)
            model.zero_grad(set_to_none=True)
            torch.cuda.empty_cache()

        per_probe[task] = probe_record

    # Done with gradient copies
    del task_grads, backup

    return {
        'batch_idx': batch_idx,
        'baseline_task_losses': baseline_task_losses,
        'baseline_focus_loss': baseline_focus_loss,
        'focus_task': focus_task,
        'per_probe': per_probe,
    }


## 4. Build model, dataloader, parameter groups

In [6]:
torch.manual_seed(SEED)
np.random.seed(SEED)

cfg = Config.fromfile(CONFIG_PATH)

if BATCH_SIZE is None:
    BATCH_SIZE = getattr(cfg.data, 'samples_per_gpu', 6)
    print(f"Auto batch_size = {BATCH_SIZE}")

cfg.data.samples_per_gpu = BATCH_SIZE
cfg.data.workers_per_gpu = min(4, BATCH_SIZE)

# Build model
print(f"Building model from {CONFIG_PATH}...")
model = build_detector(cfg.model, train_cfg=cfg.get('train_cfg'), test_cfg=cfg.get('test_cfg'))
model.init_weights()

if cfg.get('fp16', None) is not None:
    wrap_fp16_model(model)

print(f"Loading checkpoint: {CHECKPOINT_PATH}")
load_checkpoint(model, CHECKPOINT_PATH, map_location='cpu')

device_id = int(DEVICE.split(':')[1]) if ':' in DEVICE else 0
model = model.to(DEVICE)
model = MMDataParallel(model, device_ids=[device_id])

# Shared param groups (same as analyze_gradient_conflict)
param_groups, group_meta = extract_probe_params(
    model, SHARED_LAYERS, LAST_N_LAYERS, PARAM_SCOPE,
)
total_groups = len(param_groups)
total_params = sum(p.numel() for grp in param_groups.values() for p in grp.values())
print(f"\nParameter groups: {total_groups} | total params: {total_params:,}")
for gk, params in list(param_groups.items())[:10]:
    n = sum(p.numel() for p in params.values())
    meta = group_meta.get(gk, {})
    print(f"  {gk:35s}: {n:>10,} params  dec={meta.get('decoder_idx')} op={meta.get('op_type')}")
if total_groups > 10:
    print(f"  ... and {total_groups - 10} more groups")

assert total_params > 0, "No parameters selected — check SHARED_LAYERS / PARAM_SCOPE."

# Dataloader
dataset = custom_build_dataset(cfg.data.train)
dataloader = DataLoader(
    dataset, batch_size=BATCH_SIZE, shuffle=True,
    num_workers=min(4, BATCH_SIZE),
    collate_fn=partial(collate, samples_per_gpu=BATCH_SIZE), drop_last=True,
)
print(f"\nDataset size: {len(dataset)} | dataloader batch_size: {BATCH_SIZE}")


Building model from /home/yongjae/e2e/HiP-AD/projects/configs/experiments/E2_E1_stage2_18ep.py...


/home/yongjae/miniconda3/envs/hipad/lib/python3.8/site-packages/mmdet/models/backbones/resnet.py:401: UserWarning: DeprecationWarning: pretrained is deprecated, please use "init_cfg" instead
  warnings.warn('DeprecationWarning: pretrained is deprecated, '
2026-04-16 11:27:02,656 - mmcv - INFO - initialize ResNet with init_cfg {'type': 'Pretrained', 'checkpoint': 'ckpts/resnet50-19c8e357.pth'}
2026-04-16 11:27:02,658 - mmcv - INFO - load model from: ckpts/resnet50-19c8e357.pth
2026-04-16 11:27:02,660 - mmcv - INFO - load checkpoint from local path: ckpts/resnet50-19c8e357.pth
2026-04-16 11:27:02,911 - mmcv - WARNING - The model and loaded state dict do not match exactly

unexpected key in source state_dict: fc.weight, fc.bias

2026-04-16 11:27:02,927 - mmcv - INFO - initialize FPN with init_cfg {'type': 'Xavier', 'layer': 'Conv2d', 'distribution': 'uniform'}
2026-04-16 11:27:03,114 - mmcv - INFO - 
img_backbone.conv1.weight - torch.Size([64, 3, 7, 7]): 
PretrainedInit: load from ckpts/r

Loading checkpoint: /home/yongjae/e2e/HiP-AD/ckpts/exp/E2_E1_stage2_18ep/iter_2344.pth
load checkpoint from local path: /home/yongjae/e2e/HiP-AD/ckpts/exp/E2_E1_stage2_18ep/iter_2344.pth

Parameter groups: 20 | total params: 5,788,672
  dec0_norm_0                        :        512 params  dec=0 op=norm
  dec0_ffn_0                         :    920,064 params  dec=0 op=ffn
  dec0_norm_1                        :        512 params  dec=0 op=norm
  dec1_norm_0                        :        512 params  dec=1 op=norm
  dec1_ffn_0                         :    920,064 params  dec=1 op=ffn
  dec1_norm_1                        :        512 params  dec=1 op=norm
  dec2_norm_0                        :        512 params  dec=2 op=norm
  dec2_ffn_0                         :    920,064 params  dec=2 op=ffn
  dec2_norm_1                        :        512 params  dec=2 op=norm
  dec3_norm_0                        :        512 params  dec=3 op=norm
  ... and 10 more groups
{'version': 'v1.0-train

## 5. Run the probe over `NUM_BATCHES` mini-batches

In [7]:
all_results: List[Dict] = []
selective_eval = not NO_SELECTIVE_EVAL

print(f"\n=== One-Step Probe Start ===")
print(f"virtual_lr={VIRTUAL_LR}  normalize_step={NORMALIZE_STEP} ({NORMALIZE_SCOPE})")
print(f"probe_tasks={PROBE_TASKS}  focus_task={FOCUS_TASK}")
print(f"num_batches={NUM_BATCHES}  batch_size={BATCH_SIZE}\n")

for batch_idx, data in enumerate(dataloader):
    if batch_idx >= NUM_BATCHES:
        break
    try:
        result = run_probe_on_batch(
            model=model,
            data=data,
            param_groups=param_groups,
            probe_tasks=PROBE_TASKS,
            focus_task=FOCUS_TASK,
            device=DEVICE,
            fp16=FP16,
            virtual_lr=VIRTUAL_LR,
            normalize_step=NORMALIZE_STEP,
            normalize_scope=NORMALIZE_SCOPE,
            selective_eval=selective_eval,
            batch_idx=batch_idx,
        )
        all_results.append(result)
        # Condensed per-batch log
        deltas = {t: r['delta_focus'] for t, r in result['per_probe'].items()}
        print(
            f"[batch {batch_idx:03d}] base_{FOCUS_TASK}={result['baseline_focus_loss']:.4f} | "
            + " ".join(f"Δ{t}={deltas[t]:+.4e}" for t in PROBE_TASKS)
        )
    except Exception as e:
        print(f"[batch {batch_idx:03d}] FAILED: {type(e).__name__}: {e}")
        traceback.print_exc()
        continue

    if (batch_idx + 1) % 5 == 0:
        torch.cuda.empty_cache()

print(f"\nProcessed {len(all_results)}/{NUM_BATCHES} batches successfully.")



=== One-Step Probe Start ===
virtual_lr=0.001  normalize_step=False (all)
probe_tasks=['det', 'map', 'motion', 'ego', 'plan']  focus_task=plan
num_batches=20  batch_size=12

[batch 000] FAILED: OutOfMemoryError: CUDA out of memory. Tried to allocate 396.00 MiB (GPU 0; 23.65 GiB total capacity; 1.16 GiB already allocated; 140.25 MiB free; 1.19 GiB reserved in total by PyTorch) If reserved memory is >> allocated memory try setting max_split_size_mb to avoid fragmentation.  See documentation for Memory Management and PYTORCH_CUDA_ALLOC_CONF


Traceback (most recent call last):
  File "/tmp/ipykernel_568344/2152071818.py", line 13, in <module>
    result = run_probe_on_batch(
  File "/tmp/ipykernel_568344/3091035116.py", line 26, in run_probe_on_batch
    loss_dict = forward_losses(model, clone_batch(data_cache), device, fp16, selective_eval)
  File "/tmp/ipykernel_568344/4121626450.py", line 45, in forward_losses
    outputs = model(img=img, **data_on_dev)
  File "/home/yongjae/miniconda3/envs/hipad/lib/python3.8/site-packages/torch/nn/modules/module.py", line 1190, in _call_impl
    return forward_call(*input, **kwargs)
  File "/home/yongjae/miniconda3/envs/hipad/lib/python3.8/site-packages/mmcv/parallel/data_parallel.py", line 51, in forward
    return super().forward(*inputs, **kwargs)
  File "/home/yongjae/miniconda3/envs/hipad/lib/python3.8/site-packages/torch/nn/parallel/data_parallel.py", line 169, in forward
    return self.module(*inputs[0], **kwargs[0])
  File "/home/yongjae/miniconda3/envs/hipad/lib/python3.8/sit

[batch 001] FAILED: OutOfMemoryError: CUDA out of memory. Tried to allocate 396.00 MiB (GPU 0; 23.65 GiB total capacity; 1.16 GiB already allocated; 140.25 MiB free; 1.19 GiB reserved in total by PyTorch) If reserved memory is >> allocated memory try setting max_split_size_mb to avoid fragmentation.  See documentation for Memory Management and PYTORCH_CUDA_ALLOC_CONF
[batch 002] FAILED: OutOfMemoryError: CUDA out of memory. Tried to allocate 396.00 MiB (GPU 0; 23.65 GiB total capacity; 1.16 GiB already allocated; 140.25 MiB free; 1.19 GiB reserved in total by PyTorch) If reserved memory is >> allocated memory try setting max_split_size_mb to avoid fragmentation.  See documentation for Memory Management and PYTORCH_CUDA_ALLOC_CONF


Traceback (most recent call last):
  File "/tmp/ipykernel_568344/2152071818.py", line 13, in <module>
    result = run_probe_on_batch(
  File "/tmp/ipykernel_568344/3091035116.py", line 26, in run_probe_on_batch
    loss_dict = forward_losses(model, clone_batch(data_cache), device, fp16, selective_eval)
  File "/tmp/ipykernel_568344/4121626450.py", line 45, in forward_losses
    outputs = model(img=img, **data_on_dev)
  File "/home/yongjae/miniconda3/envs/hipad/lib/python3.8/site-packages/torch/nn/modules/module.py", line 1190, in _call_impl
    return forward_call(*input, **kwargs)
  File "/home/yongjae/miniconda3/envs/hipad/lib/python3.8/site-packages/mmcv/parallel/data_parallel.py", line 51, in forward
    return super().forward(*inputs, **kwargs)
  File "/home/yongjae/miniconda3/envs/hipad/lib/python3.8/site-packages/torch/nn/parallel/data_parallel.py", line 169, in forward
    return self.module(*inputs[0], **kwargs[0])
  File "/home/yongjae/miniconda3/envs/hipad/lib/python3.8/sit

[batch 003] FAILED: OutOfMemoryError: CUDA out of memory. Tried to allocate 396.00 MiB (GPU 0; 23.65 GiB total capacity; 1.16 GiB already allocated; 140.25 MiB free; 1.19 GiB reserved in total by PyTorch) If reserved memory is >> allocated memory try setting max_split_size_mb to avoid fragmentation.  See documentation for Memory Management and PYTORCH_CUDA_ALLOC_CONF


...

[batch 012] FAILED: OutOfMemoryError: CUDA out of memory. Tried to allocate 396.00 MiB (GPU 0; 23.65 GiB total capacity; 966.51 MiB already allocated; 366.25 MiB free; 994.00 MiB reserved in total by PyTorch) If reserved memory is >> allocated memory try setting max_split_size_mb to avoid fragmentation.  See documentation for Memory Management and PYTORCH_CUDA_ALLOC_CONF


Traceback (most recent call last):
  File "/tmp/ipykernel_568344/2152071818.py", line 13, in <module>
    result = run_probe_on_batch(
  File "/tmp/ipykernel_568344/3091035116.py", line 26, in run_probe_on_batch
    loss_dict = forward_losses(model, clone_batch(data_cache), device, fp16, selective_eval)
  File "/tmp/ipykernel_568344/4121626450.py", line 45, in forward_losses
    outputs = model(img=img, **data_on_dev)
  File "/home/yongjae/miniconda3/envs/hipad/lib/python3.8/site-packages/torch/nn/modules/module.py", line 1190, in _call_impl
    return forward_call(*input, **kwargs)
  File "/home/yongjae/miniconda3/envs/hipad/lib/python3.8/site-packages/mmcv/parallel/data_parallel.py", line 51, in forward
    return super().forward(*inputs, **kwargs)
  File "/home/yongjae/miniconda3/envs/hipad/lib/python3.8/site-packages/torch/nn/parallel/data_parallel.py", line 169, in forward
    return self.module(*inputs[0], **kwargs[0])
  File "/home/yongjae/miniconda3/envs/hipad/lib/python3.8/sit

[batch 013] FAILED: OutOfMemoryError: CUDA out of memory. Tried to allocate 396.00 MiB (GPU 0; 23.65 GiB total capacity; 793.32 MiB already allocated; 366.25 MiB free; 994.00 MiB reserved in total by PyTorch) If reserved memory is >> allocated memory try setting max_split_size_mb to avoid fragmentation.  See documentation for Memory Management and PYTORCH_CUDA_ALLOC_CONF
[batch 014] FAILED: OutOfMemoryError: CUDA out of memory. Tried to allocate 396.00 MiB (GPU 0; 23.65 GiB total capacity; 793.05 MiB already allocated; 366.25 MiB free; 994.00 MiB reserved in total by PyTorch) If reserved memory is >> allocated memory try setting max_split_size_mb to avoid fragmentation.  See documentation for Memory Management and PYTORCH_CUDA_ALLOC_CONF


Traceback (most recent call last):
  File "/tmp/ipykernel_568344/2152071818.py", line 13, in <module>
    result = run_probe_on_batch(
  File "/tmp/ipykernel_568344/3091035116.py", line 26, in run_probe_on_batch
    loss_dict = forward_losses(model, clone_batch(data_cache), device, fp16, selective_eval)
  File "/tmp/ipykernel_568344/4121626450.py", line 45, in forward_losses
    outputs = model(img=img, **data_on_dev)
  File "/home/yongjae/miniconda3/envs/hipad/lib/python3.8/site-packages/torch/nn/modules/module.py", line 1190, in _call_impl
    return forward_call(*input, **kwargs)
  File "/home/yongjae/miniconda3/envs/hipad/lib/python3.8/site-packages/mmcv/parallel/data_parallel.py", line 51, in forward
    return super().forward(*inputs, **kwargs)
  File "/home/yongjae/miniconda3/envs/hipad/lib/python3.8/site-packages/torch/nn/parallel/data_parallel.py", line 169, in forward
    return self.module(*inputs[0], **kwargs[0])
  File "/home/yongjae/miniconda3/envs/hipad/lib/python3.8/sit

KeyboardInterrupt: 

## 6. Aggregate statistics

For each probe task, aggregate across the N successful mini-batches:
mean ± std ± median of $\Delta L_{\text{focus}}$, plus harmful / helpful ratios.

In [ ]:
def _safe_stat(vals, fn, default=float('nan')):
    vals = [v for v in vals if v is not None and not (isinstance(v, float) and math.isnan(v))]
    return fn(vals) if vals else default


def aggregate_probe_results(results: List[Dict], probe_tasks: List[str],
                            focus_task: str) -> Dict:
    summary = {
        'focus_task': focus_task,
        'num_batches': len(results),
        'per_probe': {},
    }
    for task in probe_tasks:
        deltas, rel_deltas, cos_vals, norms = [], [], [], []
        fails = 0
        for r in results:
            pr = r['per_probe'].get(task, {})
            if pr.get('failed'):
                fails += 1
                continue
            deltas.append(pr.get('delta_focus', float('nan')))
            rel_deltas.append(pr.get('rel_delta_focus', float('nan')))
            cos_vals.append(pr.get('cos_vs_focus', float('nan')))
            norms.append(pr.get('grad_norm', float('nan')))

        clean = [d for d in deltas if not math.isnan(d)]
        harmful = sum(1 for d in clean if d > 0)
        helpful = sum(1 for d in clean if d < 0)
        n_valid = len(clean)

        summary['per_probe'][task] = {
            'n_valid': n_valid,
            'n_failed': fails,
            'mean_delta_focus':     _safe_stat(deltas, lambda v: float(np.mean(v))),
            'std_delta_focus':      _safe_stat(deltas, lambda v: float(np.std(v))),
            'median_delta_focus':   _safe_stat(deltas, lambda v: float(np.median(v))),
            'mean_rel_delta_focus': _safe_stat(rel_deltas, lambda v: float(np.mean(v))),
            'std_rel_delta_focus':  _safe_stat(rel_deltas, lambda v: float(np.std(v))),
            'mean_cos_vs_focus':    _safe_stat(cos_vals, lambda v: float(np.mean(v))),
            'mean_grad_norm':       _safe_stat(norms, lambda v: float(np.mean(v))),
            'harmful_ratio':        (harmful / n_valid) if n_valid else float('nan'),
            'helpful_ratio':        (helpful / n_valid) if n_valid else float('nan'),
        }
    return summary


summary = aggregate_probe_results(all_results, PROBE_TASKS, FOCUS_TASK)

# Console summary ------------------------------------------------------------
print(f"\n[One-Step Probe Summary] focus={FOCUS_TASK}  "
      f"batches={summary['num_batches']}  "
      f"lr={VIRTUAL_LR}  normalize={NORMALIZE_STEP}({NORMALIZE_SCOPE})")
print("-" * 100)
header = f"{'task':<8} {'n_valid':>7} {'mean_Δ':>12} {'std_Δ':>12} {'med_Δ':>12} "\
         f"{'mean_rel_Δ':>12} {'cos_vs_focus':>13} {'harm_r':>7} {'help_r':>7} {'|g|':>12}"
print(header)
print("-" * 100)
for task in PROBE_TASKS:
    s = summary['per_probe'][task]
    print(f"{task:<8} {s['n_valid']:>7d} "
          f"{s['mean_delta_focus']:>+12.4e} {s['std_delta_focus']:>12.4e} "
          f"{s['median_delta_focus']:>+12.4e} {s['mean_rel_delta_focus']:>+12.4e} "
          f"{s['mean_cos_vs_focus']:>+13.4f} "
          f"{s['harmful_ratio']:>7.2%} {s['helpful_ratio']:>7.2%} "
          f"{s['mean_grad_norm']:>12.4e}")
print("-" * 100)


## 7. Save CSV & JSON

In [ ]:
os.makedirs(OUTPUT_DIR, exist_ok=True)

# ---------- per-batch CSV ---------------------------------------------------
if SAVE_CSV and SAVE_PER_BATCH and all_results:
    csv_path = os.path.join(OUTPUT_DIR, 'one_step_probe_per_batch.csv')
    fields = ['batch_idx', f'baseline_{FOCUS_TASK}_loss']
    for t in TASK_GROUPS.keys():
        fields.append(f'baseline_{t}_loss')
    for t in PROBE_TASKS:
        fields += [
            f'delta_{FOCUS_TASK}_from_{t}',
            f'rel_delta_{FOCUS_TASK}_from_{t}',
            f'focus_loss_after_{t}',
            f'grad_norm_{t}',
            f'cos_{t}_vs_{FOCUS_TASK}',
            f'failed_{t}',
        ]
    with open(csv_path, 'w', newline='') as f:
        writer = csv.DictWriter(f, fieldnames=fields)
        writer.writeheader()
        for r in all_results:
            row = {
                'batch_idx': r['batch_idx'],
                f'baseline_{FOCUS_TASK}_loss': r['baseline_focus_loss'],
            }
            for t, v in r['baseline_task_losses'].items():
                row[f'baseline_{t}_loss'] = v
            for t in PROBE_TASKS:
                pr = r['per_probe'].get(t, {})
                row[f'delta_{FOCUS_TASK}_from_{t}']     = pr.get('delta_focus')
                row[f'rel_delta_{FOCUS_TASK}_from_{t}'] = pr.get('rel_delta_focus')
                row[f'focus_loss_after_{t}']            = pr.get('focus_loss_after')
                row[f'grad_norm_{t}']                   = pr.get('grad_norm')
                row[f'cos_{t}_vs_{FOCUS_TASK}']         = pr.get('cos_vs_focus')
                row[f'failed_{t}']                      = int(bool(pr.get('failed')))
            writer.writerow(row)
    print(f"Saved per-batch CSV: {csv_path}")

# ---------- summary JSON ----------------------------------------------------
if SAVE_JSON:
    json_path = os.path.join(OUTPUT_DIR, 'one_step_probe_summary.json')
    payload = {
        'config': {
            'config_path':    CONFIG_PATH,
            'checkpoint':     CHECKPOINT_PATH,
            'num_batches':    NUM_BATCHES,
            'batch_size':     BATCH_SIZE,
            'shared_layers':  SHARED_LAYERS,
            'last_n_layers':  LAST_N_LAYERS,
            'param_scope':    PARAM_SCOPE,
            'probe_tasks':    PROBE_TASKS,
            'focus_task':     FOCUS_TASK,
            'virtual_lr':     VIRTUAL_LR,
            'normalize_step': NORMALIZE_STEP,
            'normalize_scope': NORMALIZE_SCOPE,
            'fp16':           FP16,
            'selective_eval': not NO_SELECTIVE_EVAL,
            'seed':           SEED,
        },
        'summary': summary,
    }
    with open(json_path, 'w') as f:
        json.dump(payload, f, indent=2, default=float)
    print(f"Saved summary JSON: {json_path}")

print(f"\nAll results saved to: {OUTPUT_DIR}")


## 8. (Optional) Quick visualization

Bar chart of mean ΔL_focus per probe task — easy to read at a glance.

In [ ]:
try:
    import matplotlib.pyplot as plt

    tasks = PROBE_TASKS
    means = [summary['per_probe'][t]['mean_delta_focus'] for t in tasks]
    stds  = [summary['per_probe'][t]['std_delta_focus']  for t in tasks]
    harm  = [summary['per_probe'][t]['harmful_ratio']    for t in tasks]

    fig, axes = plt.subplots(1, 2, figsize=(12, 4))

    colors = ['#e74c3c' if m > 0 else '#2ecc71' for m in means]
    axes[0].bar(tasks, means, yerr=stds, capsize=5, color=colors, alpha=0.8)
    axes[0].axhline(0, color='k', lw=0.8)
    axes[0].set_ylabel(f'mean Δ{FOCUS_TASK} loss')
    axes[0].set_title(f'One-step virtual update → Δ{FOCUS_TASK} loss  '
                      f'(lr={VIRTUAL_LR}, normalize={NORMALIZE_STEP})')
    for i, m in enumerate(means):
        axes[0].text(i, m, f'{m:+.2e}', ha='center',
                     va='bottom' if m >= 0 else 'top', fontsize=9)

    axes[1].bar(tasks, harm, color='#e67e22', alpha=0.8)
    axes[1].axhline(0.5, ls='--', color='k', lw=0.8)
    axes[1].set_ylim(0, 1)
    axes[1].set_ylabel(f'harmful ratio (Δ{FOCUS_TASK} > 0)')
    axes[1].set_title('Fraction of batches where probe is harmful')
    for i, h in enumerate(harm):
        axes[1].text(i, h, f'{h:.0%}', ha='center', va='bottom', fontsize=9)

    plt.tight_layout()
    plot_path = os.path.join(OUTPUT_DIR, 'one_step_probe_summary.png')
    plt.savefig(plot_path, dpi=150, bbox_inches='tight')
    plt.show()
    print(f"Saved plot: {plot_path}")
except Exception as e:
    print(f"Plotting skipped: {e}")


## 9. Usage summary

### Files created / modified
| File | Role |
|---|---|
| `tools/one_step_interference_probe.ipynb` | **New** — this notebook; end-to-end one-step probe pipeline. |
| `tools/analyze_gradient_conflict.py` | **Unchanged** — imported as helper module (`TASK_GROUPS`, `_sum_task_loss`, `_selective_eval`, `move_to_device`, `get_shared_parameters_grouped`, `compute_task_gradient_grouped`, `compute_cosine_similarity`). |

### Reused helpers
- `TASK_GROUPS` — loss-prefix taxonomy for det/map/motion/ego/plan (+ det_kd).
- `_sum_task_loss` — sums losses belonging to a task.
- `_selective_eval` — context manager: Dropout / BN / `DeformableFeatureAggregation` → eval, everything else → train.
- `move_to_device` — recursive tensor move.
- `get_shared_parameters_grouped` — shared decoder param groups (backbone/neck/norm/ffn/fc_*).
- `compute_task_gradient_grouped` — per-group gradient via a single `autograd.grad` call with `retain_graph` chaining.
- `compute_cosine_similarity` — for `cos_vs_focus`.

### New helpers in this notebook
- `extract_probe_params`, `flatten_param_group`, `group_boundaries_from` — parameter selection + layout.
- `backup_params`, `restore_params` — safe checkpoint / rollback without optimizer state.
- `apply_virtual_step` — in-place `p ← p − α·g` (raw or normalized, all-or-per-group scope).
- `_sum_task_loss_nograd` — variant that works on recompute outputs that may not carry grad.
- `clone_batch`, `forward_losses`, `forward_losses_nograd`, `extract_task_loss_values` — batch reuse and forward plumbing.
- `run_probe_on_batch` — the end-to-end per-batch before/virtual-step/after routine with guaranteed restore.
- `aggregate_probe_results` — mean / std / median / harmful-ratio aggregation.

### How to run
1. Open this notebook from the HiP-AD repo root (or from `tools/`).
2. Edit **§1 Configuration** — at minimum `CONFIG_PATH`, `CHECKPOINT_PATH`, `NUM_BATCHES`, `VIRTUAL_LR`.
3. Run cells top to bottom.

Equivalent CLI-style invocation (for reference / future port to a `.py` script):

```bash
python tools/analyze_gradient_conflict.py \
    --config projects/configs/hipad_nusc_stage2_pcgrad.py \
    --checkpoint work_dirs/hipad_nusc_stage2_pcgrad/latest.pth \
    --num-batches 20 --batch-size 6 --device cuda:0 --fp16 \
    --shared-layers backbone neck norm ffn fc_before fc_after \
    --focus-task plan --probe-tasks det map motion ego plan \
    --virtual-lr 1e-3 \
    --output-dir gradient_analysis_results/one_step_probe
#   (add --normalize-step and --no-selective-eval if desired)
```

### Output columns

**`one_step_probe_per_batch.csv`** — one row per mini-batch:

| Column | Meaning |
|---|---|
| `batch_idx` | Zero-based index of the mini-batch in the dataloader stream. |
| `baseline_{focus}_loss` | $L_{\text{focus}}(B, \theta)$ before any virtual step. |
| `baseline_{t}_loss` | Baseline summed task loss for each `t ∈ TASK_GROUPS` (det, det_kd, map, motion, ego, plan). |
| `delta_{focus}_from_{t}` | $L_{\text{focus}}(B, \theta - \alpha g_t) - L_{\text{focus}}(B, \theta)$. **Key metric.** |
| `rel_delta_{focus}_from_{t}` | Same, divided by $\lvert L_{\text{focus}}(B, \theta)\rvert + \epsilon$. |
| `focus_loss_after_{t}` | Raw $L_{\text{focus}}$ after probe-task `t`'s virtual step. |
| `grad_norm_{t}` | $\lVert g_t \rVert$ on shared params (before any normalization). |
| `cos_{t}_vs_{focus}` | Full-vector cosine between $g_t$ and $g_{\text{focus}}$ on shared params. |
| `failed_{t}` | `1` if that probe-task's step failed (null/zero gradient or runtime error). |

**`one_step_probe_summary.json`** — `{config, summary}` where `summary.per_probe[t]` contains:

| Field | Meaning |
|---|---|
| `n_valid`, `n_failed` | Counts of successful / failed probes across batches. |
| `mean_delta_focus`, `std_delta_focus`, `median_delta_focus` | Central tendency and dispersion of $\Delta L_{\text{focus}}$. |
| `mean_rel_delta_focus`, `std_rel_delta_focus` | Same, normalized by baseline magnitude. |
| `mean_cos_vs_focus` | Mean cosine with focus-task gradient. |
| `mean_grad_norm` | Mean $\lVert g_t \rVert$. |
| `harmful_ratio` | Fraction of mini-batches with $\Delta L_{\text{focus}} > 0$. |
| `helpful_ratio` | Fraction with $\Delta L_{\text{focus}} < 0$. |

### Design notes
- Baseline forward uses `autograd.grad` with `retain_graph=True` across all probe tasks so we pay for the expensive forward only once per mini-batch.
- The *recompute* forward after each virtual step is under `torch.no_grad()` (cheaper, leak-free).
- Parameter backup lives on the *same device as the param* but is `detach().clone()`ed, so no grad graph is retained. GPU memory overhead ≈ one extra copy of the shared param set.
- `optimizer.step()` is never called. AdamW moments are never touched.
- `restore_params` runs inside a `try/finally`, so even if an exception hits during the recompute, $\theta$ is rolled back before the next probe task.
- Same mini-batch is reused via `clone_batch()` to guarantee before/after determinism; dropout/BN state is frozen via `_selective_eval`.
